# Phase 06A.06 — Full LoRA rank ablation
Runs independent F1 two-stage training for r8/r32/r64 and reuses the canonical r16 artifact when its locked protocol matches. No rank is called optimal in smoke scope.

In [ ]:
import gc,json,os,sys,time
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration

In [ ]:
RUN_SCOPE='smoke'; RANKS=[8] if RUN_SCOPE=='smoke' else [8,16,32,64]; ALPHAS={8:16,16:32,32:64,64:128}
EPOCHS=2; GRADIENT_ACCUMULATION=16; LEARNING_RATE=1e-4; WEIGHT_DECAY=0.01; MAX_GRAD_NORM=1.0; SMOKE_LIMIT=20; SMOKE_MAX_STEPS=2
INNER_DIR=PROJECT_ROOT/'data/splits/phase06a_inner'; TRAIN_CSV=PROJECT_ROOT/'data/splits/phase01/train.csv'; VALIDATION_CSV=PROJECT_ROOT/'data/splits/phase01/validation.csv'; VALIDATION_IDS=PROJECT_ROOT/'data/splits/phase01/validation_sample_ids.json'
OUTPUT_ROOT=PROJECT_ROOT/'outputs/phase06a/rank_ablation'; R16_DIR=PROJECT_ROOT/'outputs/phase06a/lora_r16_training'/RUN_SCOPE
protocol=Phase06AProtocol(run_scope=RUN_SCOPE); checkpoint_protocol=json.loads((INNER_DIR/'checkpoint_protocol.json').read_text(encoding='utf-8'))

## Input gates

In [ ]:
train_fit=pd.read_csv(INNER_DIR/'train_fit.csv'); inner_dev=pd.read_csv(INNER_DIR/'inner_dev.csv'); all_train=pd.read_csv(TRAIN_CSV)
assert not (set(train_fit.group_id.astype(str)) & set(inner_dev.group_id.astype(str)))
if RUN_SCOPE=='full': assert len(all_train)==EXPECTED_TRAIN_ROWS and all_train.group_id.nunique()==EXPECTED_TRAIN_GROUPS and sha256_file(VALIDATION_IDS)==EXPECTED_VALIDATION_IDS_SHA256
calibration_train=train_fit.head(SMOKE_LIMIT) if RUN_SCOPE=='smoke' else train_fit; calibration_dev=inner_dev.head(SMOKE_LIMIT) if RUN_SCOPE=='smoke' else inner_dev; final_train=all_train.head(SMOKE_LIMIT) if RUN_SCOPE=='smoke' else all_train; eval_limit=SMOKE_LIMIT if RUN_SCOPE=='smoke' else None

## Train/reuse and evaluate each rank

In [ ]:
from peft import PeftModel
results=[]
for rank in RANKS:
    alpha=ALPHAS[rank]; rank_dir=OUTPUT_ROOT/f'r{rank}'/RUN_SCOPE; rank_dir.mkdir(parents=True,exist_ok=True)
    if rank==16:
        r16_config=json.loads((R16_DIR/'config.json').read_text(encoding='utf-8')); assert r16_config['rank']==16 and r16_config['alpha']==32 and r16_config['total_tile_budget']==protocol.total_tile_budget
        adapter_dir=R16_DIR/'final_adapter'; locked_step=r16_config['locked_optimizer_step']; training_result={'reused_phase06a_04':True,'trainable_parameters':None,'peak_vram_gib':None}
    else:
        stage_a_model,tokenizer=create_lora_model(rank,alpha,training=True)
        stage_a=run_lora_training_stage(stage_a_model,tokenizer,calibration_train,output_dir=rank_dir/'stage_a_inner_selection',total_tile_budget=protocol.total_tile_budget,epochs=1 if RUN_SCOPE=='smoke' else EPOCHS,gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=SMOKE_MAX_STEPS if RUN_SCOPE=='smoke' else None,evaluation_frame=calibration_dev,evaluation_steps=1 if RUN_SCOPE=='smoke' else checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
        locked_step=stage_a['best_step']; assert locked_step
        del stage_a_model; gc.collect(); torch.cuda.empty_cache(); seed_everything(SEED)
        final_model,tokenizer=create_lora_model(rank,alpha,training=True)
        stage_b=run_lora_training_stage(final_model,tokenizer,final_train,output_dir=rank_dir/'stage_b_final_retrain',total_tile_budget=protocol.total_tile_budget,epochs=max(EPOCHS,2),gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=locked_step,evaluation_frame=None,evaluation_steps=checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
        adapter_dir=rank_dir/'final_adapter'; final_model.save_pretrained(adapter_dir); tokenizer.save_pretrained(adapter_dir); training_result=stage_b
        del final_model; gc.collect(); torch.cuda.empty_cache()
    # Frozen validation is opened only after this rank's checkpoint is locked.
    val_df=pd.read_csv(VALIDATION_CSV); frozen_ids=json.loads(VALIDATION_IDS.read_text(encoding='utf-8')); assert sorted(val_df.sample_id.astype(str))==sorted(map(str,frozen_ids))
    if RUN_SCOPE=='full': assert len(val_df)==EXPECTED_VALIDATION_ROWS
    base,tokenizer=load_model_and_tokenizer(training=False,attn_implementation=protocol.attention_implementation); base.img_context_token_id=tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN); model=PeftModel.from_pretrained(base,adapter_dir); model.eval()
    predictions,metrics=evaluate_fixed_budget_rows(model,tokenizer,val_df,frame_count=1,total_tile_budget=protocol.total_tile_budget,limit=eval_limit)
    expected_ids=val_df.head(eval_limit).sample_id if eval_limit else frozen_ids; integrity=validate_prediction_artifact(predictions,expected_ids,run_scope=RUN_SCOPE)
    predictions.to_csv(rank_dir/'predictions.csv',index=False); config={**protocol.to_dict(),'rank':rank,'alpha':alpha,'alpha_over_rank':alpha/rank,'locked_optimizer_step':locked_step,'adapter_path':str(adapter_dir),'reused_phase06a_04':rank==16}; save_json(rank_dir/'config.json',config); save_json(rank_dir/'metrics.json',metrics)
    write_run_manifest(rank_dir/'run_manifest.json',config=config,artifacts=[rank_dir/'predictions.csv',rank_dir/'metrics.json',adapter_dir/'adapter_config.json',adapter_dir/'adapter_model.safetensors'])
    results.append({'rank':rank,'alpha':alpha,'locked_step':locked_step,'accuracy':metrics['accuracy'],'macro_f1':metrics['macro_f1'],'parse_rate':metrics['parse_rate'],'trainable_parameters':training_result.get('trainable_parameters'),'peak_vram_gib':training_result.get('peak_vram_gib'),'reused_phase06a_04':rank==16})
    del model,base,val_df,frozen_ids; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for result in results:
    rank=result['rank']; rank_dir=OUTPUT_ROOT/f'r{rank}'/RUN_SCOPE
    prediction_table=pd.read_csv(rank_dir/'predictions.csv'); result['inference_duration_seconds']=float(prediction_table.latency_seconds.sum())
    training_result_path=(R16_DIR if rank==16 else rank_dir)/'stage_b_final_retrain/training_result.json'
    locked_training=json.loads(training_result_path.read_text(encoding='utf-8')); result['training_duration_seconds']=locked_training.get('elapsed_seconds'); result['peak_vram_gib']=locked_training.get('peak_vram_gib'); result['trainable_parameters']=locked_training.get('trainable_parameters')

## Summary and status

In [ ]:
summary=pd.DataFrame(results).sort_values('rank'); summary.to_csv(OUTPUT_ROOT/f'rank_summary_{RUN_SCOPE}.csv',index=False)
status='complete' if RUN_SCOPE=='full' and set(summary['rank'])=={8,16,32,64} else 'smoke_complete'
save_json(OUTPUT_ROOT/f'PHASE06A_06_STATUS_{RUN_SCOPE}.json',{'phase':'06A.06','status':status,'ranks':summary['rank'].tolist(),'rows_per_experiment':EXPECTED_VALIDATION_ROWS if eval_limit is None else eval_limit})
display(summary)

## Interpretation constraint
Rank conclusions require all four full arms and Phase 06A.07 multiplicity-aware paired analysis.